# DeepChemStable SMILES Mapping

This notebook maps `DeepChemStable` prediction CSV files to `SMILES` by using the corresponding `Tox21` CSV files.

Assumption: for the same `BatchCode` and `SliceNN`, the rows in `DeepChemStable` and `Tox21` are in the same molecule order. The `DeepChemStable` `substance_id` column is replaced by the corresponding `Tox21` `SMILES` column.

Original CSV files are not modified. New parquet files are written to a separate output directory only when `RUN_MAPPING = True`.

In [ ]:
from pathlib import Path
import re
import shutil

import pandas as pd
from tqdm.notebook import tqdm

# Input directories.
DEEPCHEM_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable")
TOX21_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21")

# Output directory. Original DeepChemStable CSV files are not changed.
OUTPUT_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES")

# Run control.
RUN_MAPPING = False
OVERWRITE_OUTPUT = False

# None means process all batches present in both DeepChemStable and Tox21.
# Example: BATCH_CODES = ["BatchE003", "BatchE004"]
BATCH_CODES = None

# Output options.
WRITE_BATCH_PARQUETS = True
WRITE_ALL_PARQUET = True
ALL_OUTPUT_PATH = OUTPUT_DIR / "DeepChemStable_All_SMILES.parquet"

# Keep these False for the clean replacement output: SMILES, Probability, Label.
ADD_BATCH_COLUMN = False
ADD_SLICE_COLUMN = False
KEEP_SUBSTANCE_ID = False

SLICE_RE = re.compile(r"^(Batch[EG]\d{3})Slice(\d+)\.csv$", re.IGNORECASE)

print(f"DeepChemStable input: {DEEPCHEM_DIR}")
print(f"Tox21 input:          {TOX21_DIR}")
print(f"Output directory:     {OUTPUT_DIR}")

## Helpers

These helpers pair DeepChemStable and Tox21 slice files by normalized slice index. For example, `Slice01` and `Slice001` both normalize to slice key `1`.

In [ ]:
def slice_key(path: Path) -> str:
    match = SLICE_RE.match(path.name)
    if not match:
        raise ValueError(f"Unexpected slice filename: {path}")
    return str(int(match.group(2)))


def collect_csv_slices(batch_dir: Path) -> dict[str, Path]:
    slices = {}
    for path in sorted(batch_dir.glob("*.csv")):
        key = slice_key(path)
        if key in slices:
            raise ValueError(f"Duplicate slice key {key} in {batch_dir}")
        slices[key] = path
    return slices


def available_batches() -> list[str]:
    deep_batches = {p.name for p in DEEPCHEM_DIR.iterdir() if p.is_dir() and p.name.startswith("Batch")}
    tox_batches = {p.name for p in TOX21_DIR.iterdir() if p.is_dir() and p.name.startswith("Batch")}
    batches = sorted(deep_batches & tox_batches)
    if BATCH_CODES is not None:
        requested = set(BATCH_CODES)
        missing = sorted(requested - set(batches))
        if missing:
            raise ValueError(f"Requested batches are not present in both inputs: {missing}")
        batches = sorted(requested)
    return batches


def validate_batch_pair(batch_code: str) -> tuple[dict[str, Path], dict[str, Path]]:
    deep_slices = collect_csv_slices(DEEPCHEM_DIR / batch_code)
    tox_slices = collect_csv_slices(TOX21_DIR / batch_code)

    deep_keys = set(deep_slices)
    tox_keys = set(tox_slices)
    if deep_keys != tox_keys:
        raise ValueError(
            f"Slice mismatch for {batch_code}: "
            f"Deep-only={sorted(deep_keys - tox_keys)}, Tox21-only={sorted(tox_keys - deep_keys)}"
        )
    return deep_slices, tox_slices


def convert_slice(batch_code: str, slice_id: str, deep_csv: Path, tox_csv: Path) -> pd.DataFrame:
    deep_df = pd.read_csv(deep_csv)
    tox_df = pd.read_csv(tox_csv, usecols=["SMILES"])

    required = {"substance_id", "Probability", "Label"}
    missing = required - set(deep_df.columns)
    if missing:
        raise ValueError(f"{deep_csv} is missing columns: {sorted(missing)}")
    if len(deep_df) != len(tox_df):
        raise ValueError(
            f"Row count mismatch for {batch_code} slice {slice_id}: "
            f"Deep={len(deep_df)}, Tox21={len(tox_df)}"
        )

    output = pd.DataFrame({
        "SMILES": tox_df["SMILES"],
        "Probability": pd.to_numeric(deep_df["Probability"], errors="coerce"),
        "Label": deep_df["Label"].astype("string"),
    })

    if KEEP_SUBSTANCE_ID:
        output.insert(1, "substance_id", deep_df["substance_id"].astype("string"))
    if ADD_SLICE_COLUMN:
        output.insert(0, "Slice", int(slice_id))
    if ADD_BATCH_COLUMN:
        output.insert(0, "BatchCode", batch_code)

    return output

## Scan And Validate

Run this before mapping to confirm the batches and slice counts that will be processed.

In [ ]:
batches = available_batches()
print(f"Batches to process: {len(batches)}")

total_slices = 0
for batch_code in batches:
    deep_slices, tox_slices = validate_batch_pair(batch_code)
    total_slices += len(deep_slices)
    print(f"{batch_code}: {len(deep_slices)} slice(s)")

print("-" * 60)
print(f"Total slices: {total_slices}")

## Run Mapping

Set `RUN_MAPPING = True` to create the mapped parquet outputs. Batch parquet files are written as `{OUTPUT_DIR}/{BatchCode}.parquet`. If enabled, all batch outputs are also merged into `DeepChemStable_All_SMILES.parquet`.

In [ ]:
def write_batch_output(batch_code: str) -> dict:
    deep_slices, tox_slices = validate_batch_pair(batch_code)
    frames = []
    rows = 0

    slice_items = sorted(deep_slices.items(), key=lambda item: int(item[0]))
    for slice_id, deep_csv in tqdm(slice_items, desc=batch_code, unit="slice", leave=False, dynamic_ncols=True):
        frame = convert_slice(batch_code, slice_id, deep_csv, tox_slices[slice_id])
        rows += len(frame)
        frames.append(frame)

    if not frames:
        raise RuntimeError(f"No slices found for {batch_code}")

    batch_df = pd.concat(frames, ignore_index=True)
    output_path = OUTPUT_DIR / f"{batch_code}.parquet"
    if output_path.exists() and not OVERWRITE_OUTPUT:
        raise FileExistsError(f"Output already exists: {output_path}")
    batch_df.to_parquet(output_path, index=False)
    return {"batch": batch_code, "rows": rows, "output_path": output_path}


def run_mapping() -> list[dict]:
    batches = available_batches()
    if OUTPUT_DIR.exists() and OVERWRITE_OUTPUT:
        # Only remove generated parquet files, not the directory itself.
        for path in OUTPUT_DIR.glob("*.parquet"):
            path.unlink()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results = []
    for batch_code in tqdm(batches, desc="Map DeepChemStable batches", unit="batch", dynamic_ncols=True):
        results.append(write_batch_output(batch_code))

    if WRITE_ALL_PARQUET:
        if ALL_OUTPUT_PATH.exists() and not OVERWRITE_OUTPUT:
            raise FileExistsError(f"Output already exists: {ALL_OUTPUT_PATH}")
        all_frames = [pd.read_parquet(item["output_path"]) for item in results]
        all_df = pd.concat(all_frames, ignore_index=True)
        all_df.to_parquet(ALL_OUTPUT_PATH, index=False)
        print(f"Wrote all-batch parquet: {ALL_OUTPUT_PATH} rows={len(all_df):,}")

    return results


if RUN_MAPPING:
    results = run_mapping()
    total_rows = sum(item["rows"] for item in results)
    print("Mapping complete.")
    print(f"Batch outputs: {len(results)}")
    print(f"Total rows:    {total_rows:,}")
    for item in results:
        print(f"{item['batch']}: rows={item['rows']:,} output={item['output_path']}")
else:
    print("Dry run only. Set RUN_MAPPING = True to create mapped parquet files.")

## Merge DeepChemStable SMILES Batch Parquets

This standalone cell vertically merges mapped `DeepChemStable_SMILES/Batch*.parquet` files into one parquet file.

It can be run by itself. Set absolute input/output paths, then set `RUN_MERGE = True`.

In [2]:
# Standalone cell: vertically merge DeepChemStable SMILES batch parquet files.
# You can run this cell without running any previous cell in this notebook.

from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# ?? Path settings: edit these absolute paths as needed ??
RUN_MERGE = True

MERGE_INPUT_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES")
MERGE_OUTPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES\DeepChemStable_All_SMILES.parquet")

# Set to True only when you intentionally want to replace an existing output file.
OVERWRITE_MERGE_OUTPUT = False

DEEPCHEMSTABLE_COLUMNS = [
    "SMILES",
    "Probability",
    "Label",
]

TARGET_SCHEMA = pa.schema([
    pa.field("SMILES", pa.large_string()),
    pa.field("Probability", pa.float64()),
    pa.field("Label", pa.large_string()),
])


def discover_batch_parquets(input_dir: Path, output_path: Path) -> list[Path]:
    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory does not exist: {input_dir}")

    files = []
    for path in sorted(input_dir.glob("Batch*.parquet")):
        if path.resolve() == output_path.resolve():
            continue
        if path.name.endswith(".tmp.parquet"):
            continue
        files.append(path)
    return files


def validate_merge_input_schema(path: Path) -> None:
    schema = pq.ParquetFile(path).schema_arrow
    missing = [column for column in DEEPCHEMSTABLE_COLUMNS if column not in schema.names]
    extra = [name for name in schema.names if name not in DEEPCHEMSTABLE_COLUMNS]
    if missing:
        raise ValueError(f"{path.name} is missing columns: {missing}")
    if extra:
        raise ValueError(f"{path.name} has unexpected columns: {extra}")


def align_merge_table(table: pa.Table) -> pa.Table:
    arrays = []
    for field in TARGET_SCHEMA:
        array = table[field.name]
        if array.type != field.type:
            array = pc.cast(array, field.type)
        arrays.append(array)
    return pa.Table.from_arrays(arrays, schema=TARGET_SCHEMA)


def merge_deepchemstable_parquets(input_paths: list[Path], output_path: Path, overwrite: bool = False) -> dict:
    if not input_paths:
        raise FileNotFoundError(f"No Batch*.parquet files found in {MERGE_INPUT_DIR}")
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Output already exists: {output_path}. Set OVERWRITE_MERGE_OUTPUT = True to replace it.")

    for path in input_paths:
        validate_merge_input_schema(path)

    temp_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_path.exists():
        if overwrite:
            temp_path.unlink()
        else:
            raise FileExistsError(f"Temporary output already exists: {temp_path}")

    writer = None
    rows_written = 0
    row_groups_written = 0

    try:
        writer = pq.ParquetWriter(
            temp_path,
            TARGET_SCHEMA,
            compression="snappy",
            use_dictionary=["SMILES", "Label"],
        )

        file_iter = tqdm(input_paths, desc="Merge DeepChemStable files", unit="file", dynamic_ncols=True)
        for path in file_iter:
            parquet_file = pq.ParquetFile(path)
            file_iter.set_postfix(file=path.name, rows=f"{parquet_file.metadata.num_rows:,}")

            row_group_iter = tqdm(
                range(parquet_file.metadata.num_row_groups),
                desc=f"{path.stem} row groups",
                unit="row group",
                leave=False,
                dynamic_ncols=True,
            )
            for row_group_index in row_group_iter:
                table = parquet_file.read_row_group(row_group_index, columns=DEEPCHEMSTABLE_COLUMNS)
                table = align_merge_table(table)
                writer.write_table(table)
                rows_written += table.num_rows
                row_groups_written += 1
                row_group_iter.set_postfix(total_rows=f"{rows_written:,}")

    finally:
        if writer is not None:
            writer.close()

    if rows_written == 0:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError("No rows were written; merge aborted.")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_path.replace(output_path)

    return {
        "output_path": str(output_path),
        "rows_written": rows_written,
        "row_groups_written": row_groups_written,
        "size_gib": output_path.stat().st_size / 1024**3,
    }


merge_input_files = discover_batch_parquets(MERGE_INPUT_DIR, MERGE_OUTPUT_PATH)
print(f"Input directory: {MERGE_INPUT_DIR}")
print(f"Output path:     {MERGE_OUTPUT_PATH}")
print(f"Input files:     {len(merge_input_files)}")

if RUN_MERGE:
    result = merge_deepchemstable_parquets(
        merge_input_files,
        MERGE_OUTPUT_PATH,
        overwrite=OVERWRITE_MERGE_OUTPUT,
    )
    print("Merge complete.")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    total_rows = 0
    for path in merge_input_files:
        metadata = pq.ParquetFile(path).metadata
        total_rows += metadata.num_rows
        print(f"{path.name:20s} rows={metadata.num_rows:>12,} row_groups={metadata.num_row_groups:>5,}")
    print("-" * 80)
    print(f"Dry run only. Set RUN_MERGE = True to create the merged parquet file.")
    print(f"Expected total rows: {total_rows:,}")


Input directory: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES
Output path:     C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES\DeepChemStable_All_SMILES.parquet
Input files:     22


Merge DeepChemStable files:   0%|          | 0/22 [00:00<?, ?file/s]

BatchE003 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

BatchE004 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

BatchE005 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

...

BatchE021 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

BatchE022 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

BatchE023 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

BatchE024 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

Merge complete.
output_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES\DeepChemStable_All_SMILES.parquet
rows_written: 8716446
row_groups_written: 22
size_gib: 0.18522691261023283


## Deduplicate DeepChemStable SMILES Parquet

This standalone cell can be run by itself. Set absolute input/output parquet paths, choose any columns for deduplication, and it writes a separate deduplicated parquet file.

The deduplication uses a temporary SQLite database on disk to keep memory low. It keeps the first row encountered for each configured deduplication key.

In [3]:
# Standalone cell: deduplicate a DeepChemStable SMILES parquet file.
# You can run this cell without running any previous cell in this notebook.

from pathlib import Path
import math
import sqlite3

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# ?? Path settings: edit these absolute paths as needed ??
RUN_DEDUP = True

DEDUP_INPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES\DeepChemStable_All_SMILES.parquet")
DEDUP_OUTPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES\DeepChemStable_All_SMILES_dedup.parquet")

# Pick any one or more columns from DEEPCHEMSTABLE_COLUMNS.
# Typical usage: ["SMILES"]
DEDUP_COLUMNS = ["SMILES"]

# Smaller batches use less memory. Increase only if your machine handles it comfortably.
DEDUP_BATCH_SIZE = 50_000

# Temporary SQLite key database. Default: next to the output parquet.
DEDUP_SQLITE_PATH = DEDUP_OUTPUT_PATH.with_name(DEDUP_OUTPUT_PATH.stem + "_seen.sqlite")

# Streaming dedup keeps the first row encountered for each dedup key.
DEDUP_KEEP = "first"

# Set to True only when you intentionally want to replace the dedup output file.
OVERWRITE_DEDUP_OUTPUT = False

# Remove the temporary SQLite key database after successful deduplication.
CLEAN_DEDUP_TEMP = True

DEEPCHEMSTABLE_COLUMNS = [
    "SMILES",
    "Probability",
    "Label",
]

TARGET_SCHEMA = pa.schema([
    pa.field("SMILES", pa.large_string()),
    pa.field("Probability", pa.float64()),
    pa.field("Label", pa.large_string()),
])


def validate_dedup_settings(columns: list[str], keep: str, batch_size: int) -> None:
    if keep != "first":
        raise ValueError("Only DEDUP_KEEP = 'first' is supported for streaming deduplication.")
    if not columns:
        raise ValueError("DEDUP_COLUMNS must contain at least one column.")
    missing = [column for column in columns if column not in DEEPCHEMSTABLE_COLUMNS]
    if missing:
        raise ValueError(f"DEDUP_COLUMNS contains unknown columns: {missing}")
    if batch_size < 1:
        raise ValueError("DEDUP_BATCH_SIZE must be at least 1.")


def validate_input_schema(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Input parquet does not exist: {path}")
    schema = pq.ParquetFile(path).schema_arrow
    missing = [column for column in DEEPCHEMSTABLE_COLUMNS if column not in schema.names]
    if missing:
        raise ValueError(f"Input parquet is missing required columns: {missing}")


def align_table(table: pa.Table) -> pa.Table:
    arrays = []
    for field in TARGET_SCHEMA:
        array = table[field.name]
        if array.type != field.type:
            array = pc.cast(array, field.type)
        arrays.append(array)
    return pa.Table.from_arrays(arrays, schema=TARGET_SCHEMA)


def _dedup_value(value):
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def _dedup_key(value_tuple: tuple) -> str:
    return repr(tuple(_dedup_value(value) for value in value_tuple))


def make_key_rows(table: pa.Table, dedup_columns: list[str]) -> list[tuple[int, str]]:
    key_columns = [table[column].to_pylist() for column in dedup_columns]
    return [(row_index, _dedup_key(values)) for row_index, values in enumerate(zip(*key_columns))]


def setup_seen_key_db(db_path: Path, overwrite: bool) -> sqlite3.Connection:
    if db_path.exists():
        if overwrite:
            db_path.unlink()
        else:
            raise FileExistsError(f"Temporary SQLite key DB already exists: {db_path}")

    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=OFF")
    conn.execute("PRAGMA synchronous=OFF")
    conn.execute("PRAGMA temp_store=FILE")
    conn.execute("PRAGMA cache_size=-200000")
    conn.execute("CREATE TABLE seen_keys (key TEXT PRIMARY KEY)")
    conn.execute("CREATE TEMP TABLE batch_keys (pos INTEGER NOT NULL, key TEXT NOT NULL)")
    conn.execute("CREATE INDEX batch_keys_key_pos_idx ON batch_keys(key, pos)")
    conn.commit()
    return conn


def find_new_positions(conn: sqlite3.Connection, key_rows: list[tuple[int, str]]) -> list[int]:
    conn.execute("DELETE FROM batch_keys")
    conn.executemany("INSERT INTO batch_keys(pos, key) VALUES (?, ?)", key_rows)

    keep_rows = conn.execute("""
        SELECT MIN(b.pos) AS pos, b.key
        FROM batch_keys b
        LEFT JOIN seen_keys s ON s.key = b.key
        WHERE s.key IS NULL
        GROUP BY b.key
    """).fetchall()

    if keep_rows:
        conn.executemany("INSERT OR IGNORE INTO seen_keys(key) VALUES (?)", [(key,) for _, key in keep_rows])
    conn.commit()
    return [pos for pos, _ in keep_rows]


def filter_positions(table: pa.Table, keep_positions: list[int]) -> pa.Table:
    if not keep_positions:
        return table.slice(0, 0)
    keep_positions.sort()
    return table.take(pa.array(keep_positions, type=pa.int64()))


def deduplicate_deepchemstable_parquet_sqlite(
    input_path: Path,
    output_path: Path,
    dedup_columns: list[str],
    batch_size: int,
    sqlite_path: Path,
    overwrite: bool = False,
    clean_temp: bool = True,
) -> dict:
    validate_dedup_settings(dedup_columns, DEDUP_KEEP, batch_size)
    validate_input_schema(input_path)
    if input_path.resolve() == output_path.resolve():
        raise ValueError("DEDUP_OUTPUT_PATH must be different from DEDUP_INPUT_PATH.")
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Dedup output already exists: {output_path}. Set OVERWRITE_DEDUP_OUTPUT = True to replace it.")

    temp_output_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_output_path.exists():
        if overwrite:
            temp_output_path.unlink()
        else:
            raise FileExistsError(f"Temporary dedup output already exists: {temp_output_path}")

    conn = setup_seen_key_db(sqlite_path, overwrite=overwrite)
    parquet_file = pq.ParquetFile(input_path)
    writer = None
    rows_read = 0
    rows_written = 0
    duplicate_rows_skipped = 0
    batches_written = 0

    total_rows = parquet_file.metadata.num_rows
    progress = tqdm(
        parquet_file.iter_batches(batch_size=batch_size, columns=DEEPCHEMSTABLE_COLUMNS),
        total=math.ceil(total_rows / batch_size),
        desc="Deduplicate DeepChemStable parquet",
        unit="batch",
        dynamic_ncols=True,
    )

    try:
        writer = pq.ParquetWriter(
            temp_output_path,
            TARGET_SCHEMA,
            compression="snappy",
            use_dictionary=["SMILES", "Label"],
        )

        for record_batch in progress:
            table = pa.Table.from_batches([record_batch])
            rows_read += table.num_rows
            table = align_table(table)

            key_rows = make_key_rows(table, dedup_columns)
            keep_positions = find_new_positions(conn, key_rows)
            duplicate_rows_skipped += table.num_rows - len(keep_positions)

            if keep_positions:
                filtered = filter_positions(table, keep_positions)
                writer.write_table(filtered)
                rows_written += filtered.num_rows
                batches_written += 1

            progress.set_postfix(
                read=f"{rows_read:,}",
                written=f"{rows_written:,}",
                skipped=f"{duplicate_rows_skipped:,}",
            )

    finally:
        progress.close()
        if writer is not None:
            writer.close()
        conn.close()

    if rows_written == 0:
        temp_output_path.unlink(missing_ok=True)
        raise RuntimeError("No rows were written; dedup aborted.")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_output_path.replace(output_path)

    if clean_temp:
        sqlite_path.unlink(missing_ok=True)

    return {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "dedup_columns": dedup_columns,
        "batch_size": batch_size,
        "sqlite_path": str(sqlite_path),
        "rows_read": rows_read,
        "rows_written": rows_written,
        "duplicate_rows_skipped": duplicate_rows_skipped,
        "batches_written": batches_written,
        "size_gib": output_path.stat().st_size / 1024**3,
    }


if RUN_DEDUP:
    result = deduplicate_deepchemstable_parquet_sqlite(
        DEDUP_INPUT_PATH,
        DEDUP_OUTPUT_PATH,
        DEDUP_COLUMNS,
        DEDUP_BATCH_SIZE,
        DEDUP_SQLITE_PATH,
        overwrite=OVERWRITE_DEDUP_OUTPUT,
        clean_temp=CLEAN_DEDUP_TEMP,
    )
    print("Dedup complete.")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    print("Dry run only. Set RUN_DEDUP = True to create the deduplicated parquet file.")
    print(f"Dedup input:    {DEDUP_INPUT_PATH}")
    print(f"Dedup output:   {DEDUP_OUTPUT_PATH}")
    print(f"Dedup columns:  {DEDUP_COLUMNS}")
    print(f"Batch size:     {DEDUP_BATCH_SIZE:,}")
    print(f"SQLite DB:      {DEDUP_SQLITE_PATH}")


Deduplicate DeepChemStable parquet:   0%|          | 0/175 [00:00<?, ?batch/s]

Dedup complete.
input_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES\DeepChemStable_All_SMILES.parquet
output_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES\DeepChemStable_All_SMILES_dedup.parquet
dedup_columns: ['SMILES']
batch_size: 50000
sqlite_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\DeepChemStable_SMILES\DeepChemStable_All_SMILES_dedup_seen.sqlite
rows_read: 8716446
rows_written: 8213623
duplicate_rows_skipped: 502823
batches_written: 175
size_gib: 0.1812648568302393
